In [ ]:
API_KEYS = {
            "faysalahmmed4200(3)@gmail.com":"hidden",
            "faysalahmmed4200@gmail.com":"hidden",
            "iffatshamiashairy@gmail.com": "hidden",
            "iffatshamiashairy(2)@gmail.com": "hidden",
           } 

In [2]:
import google.generativeai as genai
import pickle, json, re
from tqdm import tqdm
import time
import base64
from io import BytesIO
from PIL import Image
import json

# === Configure Gemini ===
genai.configure(api_key=API_KEYS["faysalahmmed4200(3)@gmail.com"])
MODEL_NAME = "gemini-2.5-flash"
model = genai.GenerativeModel(MODEL_NAME)

In [3]:
def pil_to_base64(pil_img, fmt=None):
    """
    Convert a PIL.Image to a base64 string (no data URI prefix).
    - fmt: optional format like 'JPEG', 'PNG'. If None, uses pil_img.format or 'PNG'.
    Returns: base64 string (str)
    """
    buffer = BytesIO()
    fmt = (fmt or pil_img.format or "PNG").upper()
    if fmt == "JPG":
        fmt = "JPEG"
    pil_img.save(buffer, format=fmt)
    buffer.seek(0)
    img_bytes = buffer.getvalue()
    return base64.b64encode(img_bytes).decode("utf-8")


In [4]:
# Pre-Finetune Evaluation
with open("pre_finetune_eval_predictions.pkl", "rb") as f:
    pre_finetune_results = pickle.load(f)

pre_finetune_gemini_scores = []
REQUEST_INTERVAL = 7  # 10 req per minute => 6 seconds per req + buffer 1 second = 7 seconds

for ex in tqdm(pre_finetune_results, desc="Scoring with Gemini..."):
    loop_start = time.time()
    try:
        image_b64 = pil_to_base64(ex["image"], fmt="JPEG")
        image_bytes = base64.b64decode(image_b64)  

        user_prompt = ex["prompt"]
        model_response = ex["model_response"]
        true_response = ex["true_response"]

        eval_prompt = f"""
                        You are an expert evaluator for medical AI systems.
                        Given:
                        - The image (attached)
                        - User Prompt: "{user_prompt}"
                        - Model Response: "{model_response}"
                        - Correct / Expected Response: "{true_response}"

                        Evaluate how accurate, detailed, and faithful the model's response is to the expected one.
                        Return output strictly as JSON:
                        {{
                        "score": <integer 0-100>,
                        "explanation": "<brief reasoning under 60 words>"
                        }}
                        """
        #print(image)
        #print(eval_prompt)
        #break
        response = model.generate_content(
            [
                {"mime_type": "image/jpeg", "data": image_bytes},
                {"text": eval_prompt}
            ]
        )

        raw = response.text.strip()

        # Parse JSON-like output
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
            except json.JSONDecodeError:
                print("⚠️ JSON parsing failed, assigning score=None")
                parsed = {"score": None, "explanation": raw}
        else:
            print("⚠️ No JSON-like output found, assigning score=None")
            parsed = {"score": None, "explanation": raw}

        pre_finetune_gemini_scores.append({
            "prompt": user_prompt,
            "model_response": model_response,
            "true_response": true_response,
            "gemini_feedback": parsed
        })
        loop_time = time.time() - loop_start
        if loop_time < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - loop_time)
        # else: loop already took longer than interval, continue immediately
        #break  # use this to process one test images (for debug)

    except Exception as e:
        print(f"⚠️ Error scoring sample: {e}")

# Compute average score (ignore None)
valid_scores = [
    d["gemini_feedback"]["score"] for d in pre_finetune_gemini_scores
    if isinstance(d["gemini_feedback"].get("score"), (int, float))
]
avg_score = sum(valid_scores) / len(valid_scores) if valid_scores else 0
print(f"\nAverage Gemini Score (For Pre-Finetune Model): {avg_score:.2f}/100")

Scoring with Gemini...:  46%|████▌     | 16/35 [02:52<03:12, 10.12s/it]

⚠️ JSON parsing failed, assigning score=None


Scoring with Gemini...: 100%|██████████| 35/35 [06:30<00:00, 11.15s/it]


Average Gemini Score (For Pre-Finetune Model): 46.03/100


In [5]:
print(json.dumps(pre_finetune_gemini_scores, indent=4, ensure_ascii=False))

[
    {
        "prompt": "You are an AI assistant specialized in model interpretability. I am providing:\n- CNN model Grad-CAM++ heatmap image\n- Model predicted class: Melanoma\n\nBased on the Grad-CAM++ heatmap, write a clear and concise 20–30 word explanation of which features the model focused on and why. Output only the explanation (no headings).",
        "model_response": "The model focused on the pigmented area with irregular borders and asymmetry, suggesting a potential melanoma.\n",
        "true_response": "The model focused on the lesion's irregular pigmentation, specifically the striking variation between light brown and central reddish-pink areas, indicative of melanoma's color variability and potential regression.",
        "gemini_feedback": {
            "score": 60,
            "explanation": "The response is accurate but lacks specificity. It identifies general features like pigmented area and irregular borders, but misses the heatmap's strong focus on specific colo

In [6]:
with open("pre_finetune_gemini_scores.json", "w", encoding="utf-8") as f:
    json.dump(pre_finetune_gemini_scores, f, indent=4, ensure_ascii=False)

print("Saved pretty JSON file: pre_finetune_gemini_scores.json")

Saved pretty JSON file: pre_finetune_gemini_scores.json


In [7]:
# Finetune Evaluation
with open("finetune_eval_predictions.pkl", "rb") as f:
    finetune_results = pickle.load(f)

finetune_gemini_scores = []

for ex in tqdm(finetune_results, desc="Scoring with Gemini..."):
    try:
        image_b64 = pil_to_base64(ex["image"], fmt="JPEG")
        image_bytes = base64.b64decode(image_b64)  
        user_prompt = ex["prompt"]
        model_response = ex["model_response"]
        true_response = ex["true_response"]

        eval_prompt = f"""
                        You are an expert evaluator for medical AI systems.
                        Given:
                        - The image (attached)
                        - User Prompt: "{user_prompt}"
                        - Model Response: "{model_response}"
                        - Correct / Expected Response: "{true_response}"

                        Evaluate how accurate, detailed, and faithful the model's response is to the expected one.
                        Return output strictly as JSON:
                        {{
                        "score": <integer 0-100>,
                        "explanation": "<brief reasoning under 60 words>"
                        }}
                        """

        response = model.generate_content(
            [
                {"mime_type": "image/jpeg", "data": image_bytes},
                {"text": eval_prompt}
            ]
        )

        raw = response.text.strip()

        # Parse JSON-like output
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
            except json.JSONDecodeError:
                print("⚠️ JSON parsing failed, assigning score=None")
                parsed = {"score": None, "explanation": raw}
        else:
            print("⚠️ No JSON-like output found, assigning score=None")
            parsed = {"score": None, "explanation": raw}

        finetune_gemini_scores.append({
            "prompt": user_prompt,
            "model_response": model_response,
            "true_response": true_response,
            "gemini_feedback": parsed
        })

    except Exception as e:
        print(f"⚠️ Error scoring sample: {e}")

# Compute average score (ignore None)
valid_scores = [
    d["gemini_feedback"]["score"] for d in finetune_gemini_scores
    if isinstance(d["gemini_feedback"].get("score"), (int, float))
]
avg_score = sum(valid_scores) / len(valid_scores) if valid_scores else 0
print(f"\nAverage Gemini Score (For Finetune Model): {avg_score:.2f}/100")

Scoring with Gemini...: 100%|██████████| 35/35 [05:40<00:00,  9.74s/it]


Average Gemini Score (For Finetune Model): 83.29/100


In [8]:
print(json.dumps(finetune_gemini_scores, indent=4, ensure_ascii=False))

[
    {
        "prompt": "You are an AI assistant specialized in model interpretability. I am providing:\n- CNN model Grad-CAM++ heatmap image\n- Model predicted class: Melanoma\n\nBased on the Grad-CAM++ heatmap, write a clear and concise 20–30 word explanation of which features the model focused on and why. Output only the explanation (no headings).",
        "model_response": "The model focused on the lesion's irregular borders and striking color variation, including dark brown, reddish, and lighter areas, consistent with melanoma's characteristic asymmetry and variegated pigmentation.",
        "true_response": "The model focused on the lesion's irregular pigmentation, specifically the striking variation between light brown and central reddish-pink areas, indicative of melanoma's color variability and potential regression.",
        "gemini_feedback": {
            "score": 85,
            "explanation": "The model accurately identified color variation and provided relevant clinic

In [9]:
with open("finetune_gemini_scores.json", "w", encoding="utf-8") as f:
    json.dump(finetune_gemini_scores, f, indent=4, ensure_ascii=False)

print("Saved pretty JSON file: finetune_gemini_scores.json")

Saved pretty JSON file: finetune_gemini_scores.json
